In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

#Simulation Parameteres - Constants (Methodology III A of paper - Table 1)

q  = 1.602176634e-19      # proton charge in Coulombs
m  = 1.67262192369e-27    # proton mass in Kg
B0 = 0.5                  # magnetic field strength in Tesla, field along +z
r0 = np.array([0.0, 0.0, 0.0]) #initial position in Meters
v0 = np.array([2.0e5, 0.0, 1.0e5])   # vx0, vy0, vz0 - perpendicular and parallel initial velocity in m/s
N_SAMPLES = 3000                 #requested output samples
N_PERIODS = 5                    #simulation duration in Tesla

#Derived

Omega = q * B0 / m                    # signed angular frequency (rad/s)-how fast the proton goes around
T = 2 * np.pi / abs(Omega)            # cyclotron period (s)- time for 1 circle
t_end = N_PERIODS * T                 # total time we simulate
v_perp = np.hypot(v0[0], v0[1])
r_L = m * v_perp / (abs(q) * B0)      # gyration radius (m)
pitch = v0[2] * T                     # helix pitch (m)

print("Omega =", Omega)
print("T =", T)
print("t_end =", t_end)
print("v_perp =", v_perp)
print("r_L =", r_L)
print("pitch =", pitch)

def rhs(t, Y):
    x, y, z, vx, vy, vz = Y
    return [vx, vy, vz, Omega * vy, -Omega * vx, 0.0]

def analytical(t):
    vx0, vy0, vz0 = v0
    c, s = np.cos(Omega * t), np.sin(Omega * t)
    x = r0[0] + vx0 / Omega * s + vy0 / Omega * (1 - c)
    y = r0[1] + vy0 / Omega * s - vx0 / Omega * (1 - c)
    z = r0[2] + vz0 * t
    vx = vx0 * c + vy0 * s
    vy = vy0 * c - vx0 * s
    vz = vz0 * np.ones_like(t)
    return np.vstack([x, y, z]), np.vstack([vx, vy, vz])


def run(rtol, atol, n=N_SAMPLES):
    t = np.linspace(0.0, t_end, n)
    sol = solve_ivp(rhs, (0.0, t_end), np.concatenate([r0, v0]),
                    method="DOP853", t_eval=t, rtol=rtol, atol=atol)
    r_num, v_num = sol.y[:3], sol.y[3:]
    r_ana, _ = analytical(t)
    err = np.linalg.norm(r_num - r_ana, axis=0)
    K = 0.5 * m * np.sum(v_num ** 2, axis=0)
    dK_rel = np.max(np.abs(K - K[0]) / K[0])
    dK_abs = np.max(np.abs(K - K[0]))
    return dict(t=t, r_num=r_num, r_ana=r_ana, err=err, K0=K[0],
                dK_rel=dK_rel, dK_abs=dK_abs, nfev=sol.nfev)


if __name__ == "__main__":
    res = run(1e-10, 1e-12)
    print(f"Omega   = {Omega:.6e} rad/s")
    print(f"T       = {T:.6e} s")
    print(f"t_end   = {t_end:.6e} s")
    print(f"r_L     = {r_L:.6e} m")
    print(f"pitch   = {pitch:.6e} m")
    print(f"e_max   = {res['err'].max():.6e} m")
    print(f"K0      = {res['K0']:.6e} J")
    print(f"dK_rel  = {res['dK_rel']:.6e}")
    print(f"dK_abs  = {res['dK_abs']:.6e} J")

    import scipy
print("NumPy", np.__version__, "SciPy", scipy.__version__)

Omega = 47894165.77971818
T = 1.311889497371794e-07
t_end = 6.559447486858969e-07
v_perp = 200000.0
r_L = 0.004175873965941261
pitch = 0.013118894973717939
Omega   = 4.789417e+07 rad/s
T       = 1.311889e-07 s
t_end   = 6.559447e-07 s
r_L     = 4.175874e-03 m
pitch   = 1.311889e-02 m
e_max   = 1.886278e-12 m
K0      = 4.181555e-17 J
dK_rel  = 5.565126e-10
dK_abs  = 2.327088e-26 J
NumPy 2.1.3 SciPy 1.16.3
